# 02 — Silver Layer: ETL Transformations

This notebook walks through the **Silver layer** ETL pipeline.

**Goal:** read raw Bronze data, apply cleansing and enrichment, validate quality,
and write clean Parquet to the Silver store.

**Silver contracts:**
- All critical columns are non-null.
- All prices are strictly positive.
- No duplicate `(ticker, trade_date)` pairs.
- `daily_return` = (close_t / close_{t-1}) − 1, per ticker.

In [ ]:
import sys
sys.path.insert(0, "..")

import polars as pl
pl.Config.set_tbl_rows(20)

## 1. Read Bronze data

In [ ]:
from d_processing.bronze.ingest import read_bronze

bronze_df = read_bronze(source="yahoo_finance")
print(f"Bronze rows: {len(bronze_df):,}  |  tickers: {bronze_df['ticker'].n_unique() if not bronze_df.is_empty() else 0}")
bronze_df.head(5)

## 2. Apply each transformation step individually

In [ ]:
from d_processing.silver.transform import (
    cast_types,
    remove_nulls,
    remove_invalid_prices,
    deduplicate,
    calculate_daily_return,
)

step1 = cast_types(bronze_df)
print("After cast_types:")
print(step1.schema)
step1.head(3)

In [ ]:
step2 = remove_nulls(step1)
step3 = remove_invalid_prices(step2)
step4 = deduplicate(step3)
step5 = calculate_daily_return(step4)

print(f"Rows after each step:")
print(f"  Bronze input  : {len(bronze_df):>7,}")
print(f"  cast_types    : {len(step1):>7,}")
print(f"  remove_nulls  : {len(step2):>7,}")
print(f"  invalid prices: {len(step3):>7,}")
print(f"  deduplicate   : {len(step4):>7,}")
print(f"  daily_return  : {len(step5):>7,}")

In [ ]:
step5.select(["ticker", "trade_date", "close_price", "daily_return"]).head(15)

## 3. Run quality checks

In [ ]:
from e_validation.quality_checks import run_silver_quality_checks

validated = run_silver_quality_checks(step5)
print("✔  All quality checks passed")

## 4. Write to Silver and verify

In [ ]:
from d_processing.silver.transform import write_silver, read_silver
from d_processing.silver.transform import transform_silver

silver_clean = transform_silver(bronze_df)
write_silver(silver_clean)

silver_df = read_silver()
print(f"Silver rows: {len(silver_df):,}")
silver_df.head(10)

## 5. Visualize daily returns

In [ ]:
import plotly.express as px

top5 = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]
plot_df = silver_df.filter(pl.col("ticker").is_in(top5)).sort("trade_date").to_pandas()

fig = px.line(
    plot_df,
    x="trade_date",
    y="daily_return",
    color="ticker",
    title="Daily Returns — Silver Layer",
    labels={"trade_date": "Date", "daily_return": "Daily Return"},
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()

In [ ]:
# Return distribution
fig2 = px.histogram(
    plot_df.dropna(subset=["daily_return"]),
    x="daily_return",
    color="ticker",
    nbins=60,
    title="Distribution of Daily Returns",
    barmode="overlay",
    opacity=0.6,
)
fig2.show()

## 6. Run full Silver pipeline

In [ ]:
from pipelines.silver_pipeline import SilverPipeline

result = SilverPipeline().run()
print(f"SilverPipeline produced {len(result):,} clean rows")